# Gemini Omni Flash 五段式换装视频生成 (Universal 5-Outfit Video Transition)

本 Notebook 根据输入的 **1 张人物图片** 和 **5 张服装图片**，使用一套**通用提示词 (Universal Prompt)** 调用 `gemini-omni-1.1-flash-preview` 生成包含 5 个片段、动作贴合服装风格且片段间过渡自然的完整视频。

> **注意**：如果输入的人物图包含公众人物/明星（如示例中的 `国内真人7-1_9x16.png` 为赵丽颖照片），Vertex AI 的 Responsible AI 安全策略会拦截并报错 `prohibited_content (prominent individuals)`。下方代码支持自动检测该拦截并平滑切换至非明星真人模特图 `gs://honor-omni-case/omni_flash_4_veo/model_non_celebrity_9x16.png` 进行验证。

In [ ]:
import base64
import google.auth
from google import genai
from google.genai import types
from google.cloud import storage
from IPython.display import HTML, Image as IPyImage, display

# 如果在 Google Colab 环境中运行，执行用户认证
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

client = genai.Client(
    vertexai=True,
    project="gcr-evaluation-hub",
    location="global",
    http_options=types.HttpOptions(
        headers={"Api-Revision": "2026-05-20"},
        timeout=1200000,
    ),
)

In [ ]:
# ==================== 输入配置与通用提示词 ====================

# 1. 人物图片 URI（若原图为明星触发 RAI 拦截，可替换为非明星人物图）
PERSON_URI = "gs://honor-omni-case/omni_flash_4_veo/国内真人7-1_9x16.png"
FALLBACK_PERSON_URI = "gs://honor-omni-case/omni_flash_4_veo/model_non_celebrity_9x16.png"

# 2. 5 张服装图片 URI
CLOTH_URIS = [
    "gs://honor-omni-case/omni_flash_4_veo/cloth1_9x16.png",
    "gs://honor-omni-case/omni_flash_4_veo/cloth2_9x16.png",
    "gs://honor-omni-case/omni_flash_4_veo/cloth3_9x16.png",
    "gs://honor-omni-case/omni_flash_4_veo/cloth4_9x16.png",
    "gs://honor-omni-case/omni_flash_4_veo/cloth5_9x16.png",
]

# 3. 通用提示词（不绑定特定人物或服装款式，适用于任意 1 人 + 5 衣 输入）
UNIVERSAL_PROMPT = """你是一位专业的时尚短视频导演。输入的第1张图片是人物模特参考图，后面的5张图片（第2张至第6张）是5套不同的服装参考图。
请生成一个包含5个连续片段的完整时尚换装视频，并将这5个片段合成为一个连贯的完整视频输出。具体要求如下：

1. 人物与画幅一致性：全程严格保持第1张图片中人物的面部五官、神态气质、发型、身材比例及画面比例（9:16竖屏）高度一致，背景与光影氛围自然协调。
2. 服装精准还原与去噪：在第1至第5个片段中，让第1张图中的人物依次穿上第2张至第6张图片中的对应服装。请精准还原每套服装的版型、颜色、面料材质及领口/袖口/内搭层次细节，同时完全忽略服装参考图中的衣架、原始背景以及任何商品促销文字、品牌Logo水印或装饰贴纸。
3. 适配服装风格的动态表现：根据每套服装的品类与穿搭风格，为人物自动设计自然、得体且贴合服装特性的展示动作与表情（如轻抚衣领、自信微笑、微侧转身展示版型、双手自然插兜或整理衣摆等）。
4. 片段间自然过渡与合成：请先生成人物分别试穿这5套服装的高清定妆图作为视觉锚点，再依次生成对应的5个动态视频片段，确保相邻片段之间的动作姿态、镜头运动和光影衔接流畅自然（例如利用连贯的转身、手部动作或平滑运镜衔接），避免生硬突兀的跳切，最终将5个片段合成为一个完整的视频输出。"""

def build_input(person_uri: str, cloth_uris: list, prompt: str) -> list:
    content = [{"type": "image", "mime_type": "image/png", "uri": person_uri}]
    for c_uri in cloth_uris:
        content.append({"type": "image", "mime_type": "image/png", "uri": c_uri})
    content.append({"type": "text", "text": prompt})
    return [{"type": "user_input", "content": content}]

In [ ]:
# ==================== 调用 Gemini Omni Flash 生成换装视频并保存至本地 Debug ====================
from pathlib import Path

try:
    interaction = client.interactions.create(
        model="gemini-omni-1.1-flash-preview",
        input=build_input(PERSON_URI, CLOTH_URIS, UNIVERSAL_PROMPT),
        timeout=1200.0,
    )
except Exception as e:
    if "prominent individuals" in str(e):
        print(f"原人物图触发公众人物/明星安全策略拦截，自动切换至非明星真人图: {FALLBACK_PERSON_URI}")
        interaction = client.interactions.create(
            model="gemini-omni-1.1-flash-preview",
            input=build_input(FALLBACK_PERSON_URI, CLOTH_URIS, UNIVERSAL_PROMPT),
            timeout=1200.0,
        )
    else:
        raise

def get_storage_client() -> storage.Client:
    try:
        creds, _ = google.auth.default()
        if hasattr(creds, "with_quota_project"):
            creds = creds.with_quota_project(None)
        return storage.Client(project=None, credentials=creds)
    except Exception:
        return storage.Client()

def transfer_gcs_movie_to_local(gcs_uri: str, local_path: str) -> bytes:
    """将 GCS 上的视频下载保存到本地文件并返回 bytes 供预览与 Debug。"""
    bucket_name, blob_name = gcs_uri[len("gs://"):].split("/", 1)
    blob = get_storage_client().bucket(bucket_name).blob(blob_name)
    blob.download_to_filename(local_path)
    print(f"[Debug GCS -> Local] 已从 {gcs_uri} 保存视频到本地: {local_path}")
    return Path(local_path).read_bytes()

video_idx = 0
for step in interaction.steps:
    step_type = step.get("type") if isinstance(step, dict) else getattr(step, "type", None)
    content = step.get("content") if isinstance(step, dict) else getattr(step, "content", None)
    if step_type == "model_output" and content:
        for part in content:
            p_type = part.get("type") if isinstance(part, dict) else getattr(part, "type", None)
            if p_type == "text":
                print(part.get("text") if isinstance(part, dict) else getattr(part, "text", ""))
            elif p_type == "video":
                video_idx += 1
                local_video_file = f"debug_omni_video_{video_idx}.mp4"
                mime_type = (part.get("mime_type", "video/mp4") if isinstance(part, dict)
                             else getattr(part, "mime_type", "video/mp4")) or "video/mp4"
                video_b64 = part.get("data") if isinstance(part, dict) else getattr(part, "data", None)
                video_uri = part.get("uri") if isinstance(part, dict) else getattr(part, "uri", None)
                if video_uri:
                    video_bytes = transfer_gcs_movie_to_local(video_uri, local_video_file)
                    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
                elif video_b64:
                    Path(local_video_file).write_bytes(base64.b64decode(video_b64))
                    print(f"[Debug Local] 已保存内嵌视频到本地: {local_video_file}")
                if video_b64:
                    display(HTML(
                        f"<video controls autoplay loop width=\"360\">"
                        f"<source src=\"data:{mime_type};base64,{video_b64}\" "
                        f"type=\"{mime_type}\"></video>"
                    ))